In [ ]:
import omicverse as ov
import scanpy as sc
import scvelo as scv

In [ ]:
ov.plot_set()

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import loompy
import os

In [ ]:
adata = sc.read_h5ad("C:\\Users\\25878\\Desktop\\IBD_成人转录组\\UC_T_IBD.h5ad")

In [ ]:
aaa= adata.obs

In [ ]:
adata.obs['Tmajor'] = adata.obs['celltype_T'].apply(
    lambda x: 'IEL' if 'IEL' in str(x) else x
)

In [ ]:
adata = adata[adata.obs['celltype_T'] != 'Immature_T']

In [ ]:
adata.obs['T_subtype'].value_counts() 

In [ ]:
adata=ov.pp.preprocess(adata,mode='shiftlog|pearson',n_HVGs=3000,)
adata.raw = adata
adata = adata[:, adata.var.highly_variable_features]
ov.pp.scale(adata)
ov.pp.pca(adata,layer='scaled',n_pcs=50)

In [ ]:
ov.utils.plot_pca_variance_ratio(adata)

In [ ]:
adata

In [ ]:
Traj=ov.single.TrajInfer(adata,basis='X_umap',use_rep='scaled|original|X_pca',n_comps=50,groupby='T_subtype')
Traj.set_origin_cells('cyclin_T')
Traj.inference(method='diffusion_map')

In [ ]:
import numpy as np
adata.obs['dpt_pseudotime']=adata.obs['dpt_pseudotime'].fillna(0)
adata.obs['dpt_pseudotime'].replace([np.inf], 1, inplace=True)
adata.obs['dpt_pseudotime'].replace([-np.inf], 0, inplace=True)

In [ ]:
ov.pl.embedding(adata,basis='X_umap',
                   color=['T_subtype','dpt_pseudotime'],
                   frameon='small',cmap='Reds')

In [ ]:
sc.pl.umap(adata, color=['dpt_pseudotime', 'T_subtype'], cmap='viridis')


In [ ]:
sc.pl.violin(
    adata,
    keys=["dpt_pseudotime"],
    groupby="T_subtype",
    rotation=90,
)


In [ ]:
sc.tl.paga(adata, groups='T_subtype')
sc.pl.paga(adata, threshold=0.05, show=True)

# 再用 paga 初始图初始化 umap
sc.tl.umap(adata, init_pos='paga')

In [ ]:
root_cells = adata.obs_names[adata.obs["celltype_T"] == "Cycling_CD8+"]
adata.uns["iroot"] = np.flatnonzero(adata.obs_names.isin(root_cells))[0]

# Diffusion map + DPT
sc.tl.diffmap(adata)     # 创建 diffusion basis
sc.tl.dpt(adata)         # 基于 diffusion map 计算伪时间

In [ ]:
sc.pl.umap(adata, color=["dpt_pseudotime", "celltype_T"], cmap="viridis", legend_loc="on data")


In [ ]:
adata = subset

In [ ]:
print(adata.obs['celltype_T'].value_counts())

In [ ]:
adata.obs['clusters'] = adata.obs['celltype_T']

In [ ]:
adata

In [ ]:
adata = sc.read_h5ad("/home/yangzy/T_TF/SCfate/CD8T_only_dpt.h5ad")

In [ ]:
print(adata.obs['celltype'].value_counts())

In [ ]:
Traj=ov.single.TrajInfer(adata,basis='X_umap',use_rep='X_pca',n_comps=50)
Traj.set_origin_cells('Cycling_CD8+')
Traj.set_terminal_cells(["IEL",'CD8+Naive','CD8+Tm','CD8+γδT'])
Traj.inference(method='palantir',num_waypoints=500)


In [ ]:
Traj.palantir_plot_pseudotime(embedding_basis='X_umap',cmap='RdBu_r',s=3)

In [ ]:
sc.pl.umap(adata,color=['palantir_pseudotime', 'celltype'])

In [ ]:
Traj.palantir_cal_branch(eps=0)


In [ ]:
ov.palantir.plot.plot_trajectory(adata, "conventional_IELs",
                                cell_color="palantir_entropy",
                                n_arrows=10,
                                color="red",
                                scanpy_kwargs=dict(cmap="RdBu_r"),
                                )

In [ ]:
adata

In [ ]:
adata.write('CD8T_only_dpt.h5ad')